# Домашнее задание 2 (10 баллов).

*Все задания ниже имеют равный вес*

Код для импорта мы написали за вас (не благодарите, нам не трудно). Дальше код будете писать вы. 

[Тут](https://habr.com/ru/companies/ruvds/articles/494720/) шпора по pandas. За основу домашнего задания взят ноутбук [отсюда](https://rutube.ru/video/f884aa6ed5f94120b7304506042fe5bb/) (не подглядывайте!).

In [3]:
import pandas as pd
import numpy as np

#### Описание данных

Автор д/з - плохой человек, который не стал переводить описание с мотивировкой, что весь DS на английском. Так что описание полей будет на английском:

1. Account ID
- Description: A unique identifier for each social media account in the dataset.
- Type: Integer
- Example: 1, 2, 3, …
2. Username
- Description: The username or handle of the social media account.
- Type: String
- Example: john_doe, tech_guru_22, fitness_freak
3. Platform
- Description: The social media platform the account is using (Instagram, Twitter, Facebook, TikTok, LinkedIn).
- Type: Categorical (String)
- Example: Instagram, Twitter, Facebook, TikTok, LinkedIn
4. Follower Count
- Description: The total number of followers the account has.
- Type: Integer
- Example: 1500, 245000, 78000
5. Posts Per Week
- Description: The average number of posts the account creates per week.
- Type: Integer
- Example: 3, 5, 7
6. Engagement Rate
- Description: The percentage of interactions (likes, comments, shares) relative to the follower count. This is a measure of how engaging the content is.
- Type: Float
- Range: 0.01 to 0.15
- Example: 0.045 (4.5% engagement rate)
7. Ad Spend (USD)
- Description: The monthly amount spent on advertising or promoting posts.
- Type: Float
- Example: 150.75, 850.00, 300.50
8. Conversion Rate
- Description: The percentage of users who take a desired action (e.g., clicking a link, signing up, etc.) after interacting with an ad.
- Type: Float
- Range: 0.01 to 0.05 (1% to 5% conversion rate)
- Example: 0.025 (2.5% conversion rate)
9. Campaign Reach
- Description: The total number of unique users reached by the user’s campaigns in a given month.
- Type: Integer
- Example: 5000, 20000, 15000

#### Задание 0

Подгрузите данные. Да-да, за чтение таблицы баллов не будет))

**Hint**: [pd.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

In [4]:
df = pd.read_csv("/Users/alisa/Documents/OMO/data.csv")

In [5]:
df = pd.read_csv("data.csv")

#### Задание 1

Колонка `Platform` содержит название различных платформ. Давайте представим, что в них есть некоторое отношение порядка. Закодируйте каждую платформу целым числом (от 0 до N) и положите этот "код" в новую колонку `Platform_Code`. Теперь вычислите корреляцию Спирмена между всеми парами колонок в датасете (результатом будет таблица корреляций). В качестве ответа выведите значение корреляции `Platform_Code` с `Engagement Rate`. Можете после вывода числа еще коротко написать, что оно означает (нет, это не оценивается).

**Hint**: [pd.factorize](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.factorize.html), [pd.DataFrame.select_dtypes](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.select_dtypes.html), [pd.DataFrame.corr](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html).

In [44]:
df["Platform_Code"] = pd.factorize(df["Platform"])[0]

num_df = df.select_dtypes(include="number")
corr_spearman = num_df.corr(method="spearman")

value = corr_spearman.loc["Platform_Code", "Engagement Rate"]
value

np.float64(0.03138169529349812)

#### Задание 2

Теперь посмотрите на столбец `Follower Count`. В нем какие-то числа. Иногда бывает полезно провести дискретизацию такого признака. Разбейте все значения в столбце на 4 группы: "Low", "Medium", "High", "Very High". Каждая группа включает в себя новые 25% данных. То есть, Low включает в себя 25% самых маленьких значений признака и так далее. Положите значения "Low", "Medium", "High" или "Very High" для каждого сэмпла датасета в новую колонку `Follower_Bin`. Теперь посчитайте среднее значение `Engagement Rate` для каждой категории из `Follower_Bin`. В качестве ответа выведите значение для категории "High".

**Hint**: [pd.qcut](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.qcut.html), [pd.groupby](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html), [pd.DataFrame.mean](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html)

In [9]:
df["Follower_Bin"] = pd.qcut(df["Follower Count"], 4, labels=["Low", "Medium", "High", "Very High"])

In [10]:
means = df.groupby("Follower_Bin")["Engagement Rate"].mean()
print(means.loc["High"])

0.08655032


/var/folders/h0/s049ydfn0pd7jdd1wzr0nltr0000gn/T/ipykernel_72742/3583727394.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = df.groupby("Follower_Bin")["Engagement Rate"].mean()


#### Задание 3

Иногда бывает полезно превратить широкую таблицу в длинную (например, для визуализаций сразу нескольких признаков на одной картинке). Да, звучит странно, но именно этим вы сейчас и займетесь. Сделайте новый датафрейм `melted_df`, в который вы поместите каждый сэмпл датасета 6 раз: по одному разу на значение из 'Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate' и 'Campaign Reach'. То есть, вы берете сэмпл из датасета (строку) и превращаете ее в 6 отдельных строк. Каждая отдельная строка в столбце `Metric` имеет имя из предложенного списка 5 признаков, а в столбце `Value` - значение данного сэмпла по этому признаку. Значение `Platform` повторяется в этих 6 строках.

Иначе говоря, 

```json
{
    "Account ID": 1,
    "Username": "harrislisa",
    "Platform": "TikTok",
    "Follower Count": 54217,
    "Posts Per Week": 3,
    "Engagement Rate": 0.0986,
    "Ad Spend (USD)": 538.1,
    "Conversion Rate": 0.049,
    "Campaign Reach": 1308,
    "Platform_Code": 0,
    "Follower_Bin": "Low"
}
```

превращается в 

```json
{
    "Platform": "TikTok",
    "Metric": "Follower Count",
    "Value": 54217,
},
{
    "Platform": "TikTok",
    "Metric": "Posts Per Week",
    "Value": 3,
}, ...
```

Для каждого уникальной пары значений (`Platform`, `Metric`) посчитайте моду среди всех значений `Value` для этой пары, результат сделайте списком и оставьте только наибольшее. В качестве ответа выведите сумму полученных мод (сумму всех значений в столбце `Value` уже после вычисления мод). Иначе говоря, выведите сумму всех мод значений для всех уникальных пар (`Platform`, `Metric`).

**Hint**: [pd.melt](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.melt.html), [pd.DataFrame.mode](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mode.html), [pd.DataFrameGroupBy.agg](https://pandas.pydata.org/docs/dev/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)

In [45]:

metrics = ['Follower Count', 'Posts Per Week', 'Ad Spend (USD)', 'Conversion Rate', 'Engagement Rate', 'Campaign Reach']

melted_df = pd.melt(df, id_vars = ["Platform"], value_vars=metrics, var_name="Metric", value_name="Value").dropna(subset=["Value"])


In [46]:
mode = melted_df.groupby(["Platform", "Metric"])["Value"].agg(lambda x: x.mode().max())

res = mode.sum()
res

np.float64(3100285.4716)

#### Задание 4

А теперь хочется посмотреть на самые популярные аккаунты на разных платформах. Для каждой платформы отсортируйте датафрейм по убыванию количества подписчиков (`Follower Count`) - да, без циклов, сразу для всех платформ сделать сортировку, а затем оставьте только первые три записи для каждой платформы - это и будут три самых популярных аккаунта для каждой платформы. В качестве ответа выведите саму таблицу и минимальное значение `Follower Count` в ней.

**Hint**: к *groupby* можно применять функции - это эквивалентно применению функции к каждой "группе" внутри groupby-объекта. Читайте [про применение apply к датафрейму после groupby](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#flexible-apply).

In [15]:
popular = (df.sort_values(['Platform', 'Follower Count'], ascending=[True, False])
           .groupby('Platform')
           .head(3))
popular

,Account ID,Username,Platform,Follower Count,Posts Per Week,Engagement Rate,Ad Spend (USD),Conversion Rate,Campaign Reach,Platform_Code,Follower_Bin
2403,2404,eric65,Facebook,999982,6,0.0642,884.06,0.0281,17312,2,Very High
7350,7351,patricknoble,Facebook,997915,3,0.0834,429.01,0.0182,25985,2,Very High
1689,1690,chavezjason,Facebook,997512,7,0.0834,993.20,0.0397,45717,2,Very High
8685,8686,alexandersamuel,Instagram,999726,3,0.0834,687.61,0.0205,11050,3,Very High
3965,3966,lrodgers,Instagram,999351,1,0.0834,565.07,0.0335,12391,3,Very High
2189,2190,jbrown,Instagram,997844,5,0.0642,505.61,0.0202,14717,3,Very High
3039,3040,toneill,LinkedIn,999055,4,0.0642,799.49,0.0174,21862,1,Very High
6359,6360,andrewgregory,LinkedIn,998968,7,0.1020,797.64,0.0351,15552,1,Very High
2159,2160,ashleycooper,LinkedIn,998925,6,0.0856,474.46,0.0156,45956,1,Very High
5838,5839,edwardthomas,TikTok,999739,7,0.0642,630.77,0.0325,35523,0,Very High


In [16]:
min_popular = popular['Follower Count'].min()
min_popular

np.int64(997512)

#### Задание 5

Хочется посчитать какую-то метрику. Мы хотим посмотреть, на отношение разности суммы подписчиков аккаунтов с высокой и низкой конверсией к суммарному охвату рекламы на каждой платформе. То есть, мы делим аккаунты на две группы: высокая и низка конверсия. Затем мы смотрим на то, на сколько сильно влияние аккаунтов с высокой конверсией по сравнению с аккаунтами с низкой конверсией. 

Давайте определим *Conversion Influence* следущим образом:

$$Conversion Influence = \frac{Total Follower\ Count (High) - Total Follower\ Count (Low)}{Total Campaign Reach (High)+Total Campaign Reach (Low)}$$

Считать эту метрику мы будет для каждой `Platform`. В этой формуле High - это значения всех сэмплов датасета, в которых `Conversion Rate` больше медианы, а `Low` - не более медианы. `Total Feature` - это суммарное количество значений `Feature` либо по `High` сэмплам, либо по `Low`.

Чтобы постоянно не пересчитывать, где High. где Low, сделайте новую колонку в датасете `Conversion_Category`. Положите в нее для каждой строки либо High, либо Low.

Выведите платформу с самым большим `Conversion Influence`.

**Hint**: данное задание не про *groupby*, а скорее про [pd.pivot_table](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html). Сделайте сводную таблицу, по которой уже можно посчитать суммы, а затем подставить их в формулы.

In [51]:
mediana = df['Conversion Rate'].median()
df['Conversion_Category'] = np.where(df['Conversion Rate'] > mediana, 'High', 'Low')

pivot_table = pd.pivot_table(df, ['Follower Count', 'Campaign Reach'], 'Platform', 'Conversion_Category', aggfunc='sum', fill_value=0)

den = pivot_table[('Campaign Reach','High')] + pivot_table[('Campaign Reach','Low')]
n = pivot_table[('Follower Count','High')] - pivot_table[('Follower Count','Low')]

In [53]:
Conv_Inf = n / den.replace(0, np.nan)
Conv_Inf_max = Conv_Inf.idxmax()
Conv_Inf_max

'Twitter'

#### Задание 6

Мы знаем, что вам понравилось считать метрики по формуле. Давайте закрепим этот успех. Теперь для каждой платформы посчитаем, на сколько эффективна реклама в разрезе трех последовательных записей в датасете. 

Для каждой платформы отсортируйте записи в порядке убывания `Posts Per Week`. Будто бы аккаунты, которые постят чаще, используют более "активные" стратегии по рекламе. Теперь посчитайте *скользущие суммы с окном 3* по `Campaign Reach` и `Ad Spend (USD)`. Скользящая сумма с окном N - это вы идете по массиву, берете все последовательные тройки записей и суммируете их. Для первых двух записей троек не найдется. Для них скользящее среднее - NaN, что нам не помешает. 

Теперь для каждого окна посчитайте 

$$Rolling Efficiency Ratio = \frac{Rolling Sum of Campaign Reach}{Rolling Sum of Ad Spend}$$

По сути, для каждого окна вы посчитаете сколько пользователе привлеклось за один доллар, потреченный на рекламу, в данном окне. Понятно, что значений будет столько, сколько окон. Нам интересно максимально значение такой эффективности для каждой платформы.

В качестве ответа выведите название платформы с наибольшей максимальной эффективность и наименьшей (два названия, не одно, не три, ровно два).

**Hint**: окна можно делать через [pd.DataFrame.rolling](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html).

In [22]:
post_per_week = df.sort_values(['Platform', 'Posts Per Week'], ascending=[True, False])

post_per_week[["rolling_reach", "rolling_spend"]] = (post_per_week.groupby('Platform')[['Campaign Reach', 'Ad Spend (USD)']]
                                                    .rolling(window=3).sum().reset_index(level=0, drop=True))


In [54]:
post_per_week['Rolling_Efficiency'] = post_per_week['rolling_reach'] / post_per_week['rolling_spend'].replace(0, np.nan)


max_ef = post_per_week.groupby('Platform')['Rolling_Efficiency'].max().dropna()

platform_max = max_ef.idxmax()
platform_min = max_ef.idxmin()

platform_max, platform_min

('Facebook', 'TikTok')

#### Задание 7

Это еще не все прекрасные функции pandas, которые мы хотим вам показать. Теперь вы посчитаете, сколько аккаунтов на каждой платформе одновременно лучшие по `Engagement Rate` и `Conversion Rate`.

Сделайте два отдельных суб-сета. В одном оставьте для каждой платфмормы один топовый аккаунт по `Engagement Rate`, в другом - по `Conversion Rate`. Соедините эти два подмножества по столбцу `Platform` так, что в одно строке есть описание сразу двух аккаунтов-лидеров. Теперь посмотрите равны ли имена аккаунтов в одной строке. Выведите количество строк, в которых названия аккаунтов совпадают.

In [35]:
eng_top = df.loc[df.groupby("Platform")["Engagement Rate"].idxmax(), ["Platform","Username"]].rename(columns={"Username":"eng_user"})

In [36]:
conv_top = df.loc[df.groupby("Platform")["Conversion Rate"].idxmax(), ["Platform","Username"]].rename(columns={"Username":"conv_user"})

In [37]:
merge = eng_top.merge(conv_top, on='Platform')

In [38]:
same_acc = (merge["eng_user"] == merge["conv_user"]).sum()
same_acc

np.int64(0)

#### Задание 8

Давайте теперь что-то попроще сделаем. Например, посчитаем отношение суммарного количества подписчиков на аккаунтах с высокой конверсией к такой же сумме в аккаунтах с низкой конверсией (очевидно, для каждой платформы). По сути, мы просто хотим получить число, которое характеризует, на сколько сильно аккаунты с высокой конверсией "доминируют" над аккаунтами с низкой конверсией в плане количества подписчиков.

Высокой конверсией будем считать конверсию больше средней. Остальное - низкая. Посчитайте суммы подписчиков для каждой платформы, поделите одно на другое и выведите разницу между самым большим значением и самым маленьким, а также платформы, которые соотвутствуют этим значениям.

Используйте магическую команду `%%time`, чтобы замерить, сколько времени ушло на исполнение вашего pandas-скрипта.

In [68]:
%%time
mean_conversion = df["Conversion Rate"].mean()
df["Conversion_Category"] = np.where(df["Conversion Rate"] > mean_conversion, "High", "Low")

pt = pd.pivot_table(
    df,
    values="Follower Count",
    index="Platform",
    columns="Conversion_Category",
    aggfunc="sum",
    fill_value=0
)

dominate = pt["High"] / pt["Low"].replace(0, np.nan)
dominate = dominate.dropna()

difference = dominate.max() - dominate.min()
max_platform = dominate.idxmax()
min_platform = dominate.idxmin()

difference, max_platform, min_platform


CPU times: user 8.18 ms, sys: 4.57 ms, total: 12.8 ms
Wall time: 10.6 ms


(np.float64(0.17688741338715763), 'Twitter', 'Instagram')

#### Задание 9

А теперь решите задание 8 чисто питоном. Никаких функций и методов pandas. Только питоновские циклы. Замерьте время выполнения кода. Наконец, сравните время в задании 8 и 9. Напишите ниже, кто же победил: чистый python и pandas?

**Hint**: Чтобы итерироваться по датафрейму, можно из него сделать генератор через [pd.DataFrame.iterrows](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iterrows.html) или [pd.DataFrame.itertuples](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.itertuples.html#pandas.DataFrame.itertuples). К слову, это не все способы итерироваться по датафрейму.

In [63]:
%%time
cols = list(df.columns)
i_platform = cols.index("Platform")
i_follow = cols.index("Follower Count")
i_conv = cols.index("Conversion Rate")

total = 0.0
n = 0
for row in df.itertuples(index=False, name=None):
    total += float(row[i_conv])
    n += 1
mean_conversion = total / n

high, low = {}, {}
for row in df.itertuples(index=False, name=None):
    platform = row[i_platform]
    follow = float(row[i_follow])
    conv = float(row[i_conv])

    if conv > mean_conversion:
        high[platform] = high.get(platform, 0.0) + follow
    else:
        low[platform] = low.get(platform, 0.0) + follow

dominate = {}
for platform in set(high) | set(low):
    l = low.get(platform, 0.0)
    dominate[platform] = (high.get(platform, 0.0) / l) if l != 0 else float("inf")

max_platform = max(dominate, key=dominate.get)
min_platform = min(dominate, key=dominate.get)
diff = dominate[max_platform] - dominate[min_platform]

diff, max_platform, min_platform


CPU times: user 18.9 ms, sys: 1.17 ms, total: 20 ms
Wall time: 19.4 ms


(0.17688741338715763, 'Twitter', 'Instagram')

В задание 8 время вышло время (Wall time:) 10.6 ms
В задании 9 время составило 19.4 ms

 19.4 / 10.6 =  1.83
Можем сделать вывод, что pandas справился примерно в 1.83 раза быстрее, чем решения только через циклы python

**А победителем является**: <А ТУТ МОЙ ОТВЕТ, Я ЗАМЕТИЛ, ЧТО В ЗАДАНИИ НУЖНО ЕЩЕ ЧТО-ТО НАПИСАТЬ ПОСЛЕ КОДА, ИНАЧЕ НЕ ПОЛУЧУ ПОЛНЫЙ БАЛЛ ЗА ЗАДАНИЕ>

#### Задание 10

Крайне серьезное задание. Отнеситесь к нему соответствующе. В ячейке ниже напишите ваш любимый анекдот или мем (только без баянов, окей?). Можно плохие. Помните, это задание на полный балл. Проверяющий работу ассистент должен улыбнуться.

Если вставляете картинку, то убедитесь, что вы ее не подгружаете локально. А то будет неудобно - потерять балл на этом задании, когда надо было выложить картинку на облако и прокинуть ссылку. И нет, нельзя сюда просто ссылку вставить. Либо ищите, как вставить картинку, либо смешной анекдот. Есть всего два стула - выбирайте...

Анекдот: Гена с Чебурашкой купили косячок.


Гена с Чебурашкой купили косячок. Приходят домой. Гена говорит:
- Чебурашка, я пойду приму ванну, помоюсь немного и потом раскурим с тобой косяк. Только ты падла не кури без меня.
Чебурашка:
- нет Гена, ты что. Я без тебя не буду.
Ну уходит Гена в ванну. Чебурашка сел телек смотреть. Сидит делать нечего, по я щику одну хрень показывают. Ну думает: "дай я хапану немного, мол незаметит." Ну и начал курить. "М.. М.. м.." И все скурил. Сидит:
- Е-мое, меня же Гена убъет. Надо беломора затолкать.
Тут голос из ванной:
- Чебурашка, приниси полотенце!
- Бля... Полотенце. Щас спалюсь. Че делать? Так... так-так-так, что надо делать. Ага. Надо пойти в комнату, открыть шкаф, взять полотенце, закрыть шкаф.
Тут опять голос из ванной:
- Чебурашка!!! Мать твою, где полотенце?
Тот весь на изводе. Собирается с мыслями. Приходит в комнату.... Стоит, думает:
- Блять, нахуй я сюда пришел?
Опять голос из ванной.
- Где полотенце?!
- А-а-а-... полотенце!!! - Подходит к шкафу, открывает дверь берет полотенце
и уходит. Там Гена уже заебался ждать. Чебурашка думает:
- как сказать ему? Ведь щас запалит. И пиздец мне будет. Так... Гена... Помнишь ты просил меня принести полотенце? Вот твое... Нет. Спалит бля. Че же делать... Геннадий, вот твое... Тьфу ты бля. почему Геннадий?"
Гена:
- Чебурашка! Ну че ты там?!
- Все... думает Чебурашка, - надо идти. Как сказать. А.. Гена на, Гена на, Гена на, Гена на ААААААА, Крокодил в ванной!!!!!!!!!!